# A_04 PyTorch in this repo

*Course appendix A: the toolbox (our own, not the book's appendix A). Dip in whenever you meet a tool you don't know yet.*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, write down what you think it will do. Then run it and compare.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll see `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → [model learns] → model writes → MCP tool
> ```
>
> You are here: **model learns**
>
> Project files this connects to:
>
> - `src/homer_ai/model.py`
> - `src/homer_ai/train_homer.py`
>
> **This notebook:** `model.py` and the training loop are short, but every line is a PyTorch tool. Here they're explained one at a time.
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- How `model.py` builds the model from PyTorch parts
- What each part does: the embedding table, linear layers, softmax
- How the loss is measured, and how the weights get adjusted
- How many weights our model has, and why

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **weight** | one adjustable number inside the model; learning means changing weights | our model has 11,840 |
| **layer** | one part of the model that turns numbers in into numbers out | `nn.Linear` |
| **embedding table** | a lookup table: one row of numbers for each token | 137 rows × 32 columns |
| **softmax** | turns any scores into chances that add up to 1 | `[2, 1] → [0.73, 0.27]` |
| **loss** | a mistake score: how wrong the guesses were; lower is better | `F.cross_entropy` |
| **gradient** | for each weight, which way is downhill: how to change it to lower the loss | worked out by `loss.backward()` |
| **optimizer** | the part that adjusts the weights using the gradients | `AdamW` |

## 3. The idea

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ What's `nn.Module`?

`nn` is PyTorch's toolbox for building models (`nn` stands for neural network). `nn.Module` is the blueprint every model starts from. Our `HomerGreekLLM` class says `class HomerGreekLLM(nn.Module)`, which means "a model, built on PyTorch's blueprint".

The blueprint keeps track of all the model's weights, so the optimizer can find and adjust them.

👉 **Predict first:** how many weights does our model have?

In [ ]:
import torch
import torch.nn.functional as F
from torch import nn

from homer_ai.model import HomerGreekLLM

torch.manual_seed(42)
model = HomerGreekLLM(vocab_size=137, embed_dim=32)
for name, w in model.named_parameters():
    print(name, tuple(w.shape))
print("total weights:", sum(w.numel() for w in model.parameters()))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

5 groups of weights, **11,840** in total. `numel` means "number of elements", so it says how many numbers a tensor holds. (`F` is PyTorch's collection of ready-made maths functions. We use it below.)

</details>

### ❓ What's `nn.Embedding`, and why is its table 137 × 32?

An **embedding table** is a lookup table with one row of numbers for each token in the vocabulary. Give it a token ID, and it hands back that row.

Its shape is rows = vocabulary size, columns = `embed_dim`. We have 137 tokens (135 characters plus 2 special tokens) and `embed_dim=32`, so the table is 137 × 32. Each token gets 32 numbers that the model learns, so that tokens used in similar ways end up with similar rows.

👉 **Predict first:** what shape comes out for 3 token IDs?

In [ ]:
table = nn.Embedding(137, 32)
ids = torch.tensor([41, 40, 36])  # π ο λ
rows = table(ids)
print(table.weight.shape)
print(rows.shape)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The table is `(137, 32)`, and 3 IDs give back 3 rows of 32 numbers: `(3, 32)`.

</details>

### ❓ What does a real row of the table look like?

In our trained model, each character's row holds 32 learned numbers. Here are the first 4 of two rows (the `…` stands for the other 28):

```
ID 58  'ἄ'  →  [-0.15, 0.21, -0.58, -0.27] …
ID 38  'ν'  →  [0.21, -0.65, 0.28, 0.13] …
```

No single number means anything on its own. Together, the 32 numbers are how the model "sees" the character.

👉 **Predict first:** will `ἄ` and `ν` have similar rows? (Nothing told the model they're related.)

In [ ]:
from homer_ai.lessons import load_homer_model

tok, saved_model = load_homer_model()  # yours if you trained one, otherwise models/
if saved_model is not None:
    learned = saved_model.embedding.weight.detach()
    for ch in ["ἄ", "ν"]:
        i = tok.encode(ch)[0]
        first4 = [round(v, 2) for v in learned[i][:4].tolist()]
        print(f"ID {i}  {ch!r}  →  {first4} …")
else:
    print("no trained model found")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Two rows of learned numbers, and they look nothing alike. `saved_model.embedding.weight` is the whole table, one row per token. In the saved file the same numbers sit under the name `embedding.weight`, inside `model_state` ([05_06](../05_training_on_the_odyssey/05_06_saving_and_loading.ipynb)).

</details>

### ❓ Can I see the rows of a real embedding table?

Yes, as a heatmap with one row per character and one column for each of the 32 numbers. Green is a positive number, pink-red a negative one, and dark is near zero.

👉 **Predict first:** will two different characters have the same colour pattern?

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import GOOD_BAD, INK, MUTED, studio_figure

if saved_model is not None:
    chars = list("ἄνδρα μοι")
    ids = [tok.encode(ch)[0] for ch in chars]
    grid = learned[ids].detach().numpy()
    limit = abs(grid).max()

    fig = studio_figure(
        "Embedding rows", f"{len(chars)} characters, 32 learned numbers each"
    )
    ax = fig.add_axes((0.13, 0.1, 0.8, 0.68))
    ax.set_facecolor("none")
    ax.imshow(grid, cmap=GOOD_BAD, vmin=-limit, vmax=limit, aspect="auto")
    ax.set_yticks(range(len(chars)), [repr(ch) for ch in chars], fontsize=13, color=INK)
    ax.set_xticks(range(0, 32, 4), [str(i) for i in range(0, 32, 4)])
    ax.tick_params(length=0)
    for side in ax.spines.values():
        side.set_visible(False)
    ax.set_xlabel("the 32 numbers in the row", color=MUTED)
    plt.show()
else:
    print("no trained model found")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Every character has its own pattern of colours, and no single square means anything alone. Don't try to read much into a single row. In a model this small the likeness between rows is faint, and [02_05](../02_text_to_numbers/02_05_token_embeddings.ipynb) measures it properly.

</details>

### ❓ What's `nn.Linear`?

A **linear layer** turns a vector of one size into a vector of another size, using a table of weights (it's matrix multiplication, from A_03). Our model's last layer, `head`, turns 32 numbers into 137 scores, one score per possible next token.

👉 **Predict first:** what shape comes out?

In [ ]:
head = nn.Linear(32, 137, bias=False)
scores = head(rows)
print(scores.shape)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`(3, 137)`: for each of the 3 positions, a score for every token in the vocabulary. These raw scores are called **logits**.

</details>

### ❓ What does `F.softmax` do?

**Softmax** turns any list of scores into chances between 0 and 1 that add up to 1. Bigger score, bigger chance.

```
scores    ι  2.0     ν  1.0     ς  0.1
             ↓ softmax
chances   ι  ?       ν  ?       ς  ?        (together: 1)
```

👉 **Predict first:** which of the three gets the biggest chance?

In [ ]:
s = torch.tensor([2.0, 1.0, 0.1])
chances = F.softmax(s, dim=-1)
print(chances)
print(chances.sum())

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

About `[0.66, 0.24, 0.10]`, adding up to 1. `dim=-1` means "along the last direction", here the row. Our model uses softmax inside attention.

</details>

### ❓ Can I see what softmax does?

Here are two bar charts side by side: the raw scores on the left, the chances softmax makes from them on the right.

👉 **Predict first:** will the order of the bars change?

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import ACCENT, INK, MUTED, studio_axes, studio_figure

letters = ["ι", "ν", "ς"]
fig = studio_figure("Softmax", "Scores in, chances out", size=(9, 5))
panels = [
    ("scores", s.tolist(), (0.08, 0.14, 0.38, 0.62), "{:.1f}"),
    ("chances", chances.tolist(), (0.56, 0.14, 0.38, 0.62), "{:.2f}"),
]
for label, values, rect, fmt in panels:
    ax = studio_axes(fig, rect)
    ax.set_axisbelow(True)  # grid lines behind the bars
    bars = ax.bar(letters, values, color=MUTED)
    bars[0].set_color(ACCENT)
    for bar, value in zip(bars, values, strict=True):
        ax.text(
            bar.get_x() + bar.get_width() / 2, value + max(values) * 0.02,
            fmt.format(value), ha="center", color=INK, family="monospace",
        )
    ax.set_ylim(0, max(values) * 1.15)
    ax.tick_params(axis="x", labelsize=14, labelcolor=INK)
    ax.grid(axis="x", visible=False)
    ax.set_title(label, color=ACCENT, fontsize=13, fontweight="bold")
plt.show()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The order stays the same, so the biggest score gets the biggest chance. But the chances are squeezed into 0 to 1 and add up to 1, so they can be read as probabilities.

</details>

### ❓ What does `F.cross_entropy` measure?

It's the **loss**: how surprised the model was by the right answer. You give it the model's scores and the correct next token. If the model gave the right token a high chance, the loss is low.

👉 **Predict first:** which case has the lower loss?

In [ ]:
right_answer = torch.tensor([0])
confident_and_right = torch.tensor([[5.0, 0.0, 0.0]])
confident_and_wrong = torch.tensor([[0.0, 5.0, 0.0]])
print(F.cross_entropy(confident_and_right, right_answer).item())
print(F.cross_entropy(confident_and_wrong, right_answer).item())

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

About `0.01` when confident and right, about `5.01` when confident and wrong. Training pushes this number down.

</details>

### ❓ What do `backward`, `zero_grad` and `step` do?

They're the three lines at the heart of every training loop:

1. `loss.backward()` works out the **gradient** for every weight, which is the way to change it to lower the loss. Working it out backwards through the model is called backpropagation.
2. `optimizer.step()`: the optimizer nudges every weight a small step downhill. Ours is AdamW, a popular one.
3. `optimizer.zero_grad()`: wipes the old gradients before the next round. Without it they pile up, like not clearing a calculator.

👉 **Predict first:** after 50 rounds, will the loss be lower than at the start?

In [ ]:
torch.manual_seed(42)
tiny = nn.Linear(3, 3, bias=False)
optimizer = torch.optim.AdamW(tiny.parameters(), lr=0.1)
x = torch.tensor([[1.0, 0.0, 0.0]])
target = torch.tensor([2])  # the right answer is token 2

for step in range(50):
    loss = F.cross_entropy(tiny(x), target)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step in (0, 49):
        print(f"round {step + 1}: loss {loss.item():.3f}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The loss falls a lot in 50 rounds. And that's all learning is: the same three lines, repeated.

</details>

## 5. Full example: one training step on real Greek

Our real model, one window of Book 1, one step: the same lines as `train()` in `train_homer.py`.

In [ ]:
from homer_ai.train_homer import BOOKS_DIR, load_data

data, tokenizer, text = load_data(BOOKS_DIR)
torch.manual_seed(42)
model = HomerGreekLLM(vocab_size=tokenizer.vocab_size, embed_dim=32)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)

inputs = data[0:24]  # 24 characters
targets = data[1:25]  # the same, shifted by one: the next character each time
for step in range(3):
    logits = model(inputs)
    loss = F.cross_entropy(logits, targets)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f"step {step + 1}: loss {loss.item():.3f}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The loss goes down a little with each step on this one window. The real training does this for every window of the book, 800 times over. The targets are the inputs shifted by one, because the job is always "guess the next character".

</details>

## 6. Recap

- `nn.Module` is the blueprint. It keeps track of the model's 11,840 weights.
- `nn.Embedding` is a lookup table (137 × 32), `nn.Linear` turns vectors into scores, and softmax turns scores into chances.
- `F.cross_entropy` is the loss. `backward` finds the gradients, `step` adjusts the weights, and `zero_grad` clears the gradients.

**Next:** [A_05 tiktoken and BPE](A_05_tiktoken_bpe.ipynb).